# Data Ingestion with Lakeflow Connect

Training notebook covering Lakeflow Connect: fully-managed connectors that ingest data from enterprise applications, databases, and cloud storage into Unity Catalog — with incremental ingestion, schema evolution, and orchestration built in.

## 📋 What We'll Cover

1. **What is Lakeflow Connect** — purpose, capabilities, and where it fits in the Databricks platform
2. **Connector Architecture** — SaaS, database, and file connector components
3. **Supported Connectors** — enterprise applications, databases, cloud storage, message buses
4. **Incremental Ingestion & CDC** — how Lakeflow Connect tracks and replays changes efficiently
5. **Orchestration** — schedules, jobs, and chaining tasks for managed connectors
6. **Cloud Files Ingestion & Default Schedule** — what the add-data UI creates, and how to view and change the schedule
7. **Table Schedule vs Job Schedule** — how they interact and how to orchestrate refreshes from a job

## 1. What is Lakeflow Connect

Lakeflow Connect is Databricks' **managed ingestion service** that moves data from external systems into Unity Catalog with minimal setup. It handles connector management, scheduling, schema evolution, and incremental change tracking so you don't need to maintain bespoke ETL code.

### Where it fits

```
 External Systems                Databricks Platform
 ┌──────────────────┐            ┌─────────────────────────────┐
 │  Salesforce      │            │                             │
 │  SAP             │──Lakeflow──▶  Unity Catalog Tables       │
 │  Workday         │  Connect   │  (Bronze / Silver / Gold)   │
 │  PostgreSQL      │            │                             │
 │  Kafka           │            └─────────────────────────────┘
 │  Azure Blob / S3 │
 └──────────────────┘
```

### Key capabilities

| Capability | Description |
|---|---|
| **Fully managed** | No infrastructure to provision or maintain |
| **Incremental ingestion** | Only changed records are loaded after the initial full sync |
| **CDC support** | Captures inserts, updates, and deletes from databases |
| **Schema evolution** | Automatically adapts when source schemas change |
| **Unity Catalog native** | Tables land directly in UC with lineage, access control, and governance |
| **Orchestration built-in** | Scheduling a managed connector pipeline in the ingestion UI creates a Lakeflow Job; cloud-files streaming tables keep their schedule on the table (see Sections 5–7) |

## 2. Connector Architecture

Lakeflow Connect has three connector families, each with slightly different components.

### SaaS & File Connectors (e.g. Salesforce, Workday, Azure Blob)

```
 ┌──────────────┐    ┌────────────────┐    ┌──────────────────────┐
 │  Connection  │───▶│ Ingestion      │───▶│ Destination Tables   │
 │ (credentials)│    │ Pipeline       │    │ in Unity Catalog     │
 └──────────────┘    └────────────────┘    └──────────────────────┘
```

* **Connection** — stores credentials and endpoint details securely (backed by Databricks secrets)
* **Ingestion Pipeline** — the Lakeflow pipeline that reads from the source and writes to UC
* **Destination Tables** — Delta tables created/updated in the chosen catalog and schema

### Database Connectors (e.g. PostgreSQL, SQL Server, MySQL)

```
 ┌──────────────┐    ┌─────────────────┐    ┌──────────────┐    ┌───────────────┐
 │  Connection  │───▶│ Ingestion       │───▶│  Staging     │───▶│  Destination  │
 │              │    │ Gateway (job)   │    │  Storage     │    │  Tables in UC │
 └──────────────┘    └─────────────────┘    └──────────────┘    └───────────────┘
```

Database connectors add an **Ingestion Gateway** (a continuous Lakeflow Job) and **staging storage** (cloud object store) to support continuous CDC.

### File / Cloud Storage Connectors

Read files directly from Azure Blob, ADLS, S3, or GCS into Delta tables. Uses Auto Loader under the hood with managed checkpointing.

## 3. Supported Connectors

### Enterprise Applications (SaaS)

| Connector | Incremental Support | Notes |
|---|---|---|
| Salesforce | Yes | Cursor-based; uses `SystemModstamp` |
| Workday | Yes | Report-based via Workday RaaS API |
| SAP | Yes | SAP OData or ABAP CDS views |
| ServiceNow | Yes | `sys_updated_on` cursor |
| Google Analytics | Yes | Data API |
| HubSpot | Yes | CRM API |

### Databases

| Connector | CDC | Notes |
|---|---|---|
| PostgreSQL | Yes | Logical replication (pgoutput) |
| MySQL | Yes | Binlog CDC |
| SQL Server | Yes | Change tracking + CDC |
| Oracle | Yes | LogMiner CDC |
| MongoDB | Yes | Change streams |

### Cloud Storage & Files

| Source | Format | Notes |
|---|---|---|
| Azure Blob / ADLS | CSV, JSON, Parquet, Avro | Auto Loader backed |
| Amazon S3 | CSV, JSON, Parquet, Avro | Auto Loader backed |
| Google Cloud Storage | CSV, JSON, Parquet | Auto Loader backed |

### Message Buses

| Connector | Notes |
|---|---|
| Apache Kafka | Structured Streaming under the hood |
| Azure Event Hubs | Kafka-compatible endpoint |
| Confluent Cloud | Managed Kafka |

> 💡 **Tip:** In the Databricks UI, navigate to **Data Engineering → Lakeflow Connect** to see the connectors available in your workspace.

## 4. Incremental Ingestion & CDC

### How Incremental Ingestion Works

```
Run 1 (full load)
 Source: 1,000,000 rows ──────────────────────────▶ UC Table: 1,000,000 rows
                          ↓ records checkpoint

Run 2 (incremental)
 Source: only 5,000 changed rows ─────────────────▶ UC Table: upserted / appended
```

**First run:** performs a full load of all selected tables/objects.
**Subsequent runs:** only records that changed since the last run are ingested.


## 5. Orchestration

When you add a schedule to a **managed ingestion pipeline** (SaaS or database connector, e.g. Salesforce, SQL Server) in the ingestion UI, Databricks creates a **Lakeflow Job** that runs the pipeline on that schedule. You can chain additional tasks to that job.

> ⚠️ This automatic job applies to **managed connector pipelines** only. A streaming table created from **cloud files** with the add-data / file upload UI does **not** get a job — its schedule is stored on the table itself. See **Section 6**.

| What you create | Where the schedule lives | Job created automatically? |
|---|---|---|
| Managed connector pipeline (SaaS / database) scheduled in the ingestion UI | Lakeflow Job | Yes |
| Lakeflow pipeline built in the pipeline editor, scheduled from the pipeline UI | Lakeflow Job | Yes |
| Streaming table from cloud files (add-data UI or `CREATE STREAMING TABLE ... SCHEDULE`) | On the table (its hidden serverless pipeline) | **No** |

### Trigger options

| Trigger | Description |
|---|---|
| **Scheduled (cron)** | Run at fixed intervals, e.g. every hour |
| **Continuous** | Keep the pipeline running permanently (streaming CDC) |
| **Manual** | Run on demand via UI, API, or CLI |

### Chaining tasks

```
Job: daily_customer_sync
 ├── Task 1: Lakeflow Connect ingestion  (ingest customers from Salesforce)
 ├── Task 2: Notebook transformation     (depends on Task 1)
 └── Task 3: Dashboard refresh           (depends on Task 2)
```

You can add extra tasks to the auto-generated job in the **Lakeflow Jobs** UI or via Declarative Automation Bundles.

## 6. Cloud Files Ingestion & Default Schedule

### Example used in this section
The table `training.bronze.customer_lakeflow_test` was created from cloud files with the **add-data / file upload UI**, reading the customer CSV files from `abfss://inboundfiles@unityms.dfs.core.windows.net/customer`.

### What the UI actually creates

| Object | Value for this example |
|---|---|
| Table type | `STREAMING_TABLE` (comment: *Created by the file upload UI*) |
| Definition | `SELECT ... FROM STREAM read_files('abfss://inboundfiles@.../customer', schemaHints => ...)` — Auto Loader |
| Refresh engine | A **hidden serverless pipeline** named `ST-training.bronze.customer_lakeflow_test`, created and managed by the system |
| Schedule | `CRON '0 0 * * * ?' AT TIME ZONE 'Europe/Amsterdam'` — stored on the table / its pipeline |
| Lakeflow Job | **None** — no job is created |

### The default schedule
In this example, the UI set the cron `0 0 * * * ?`, which means **every hour, on the hour** (not daily). Quartz cron fields are: `seconds minutes hours day-of-month month day-of-week`.

| Cron | Meaning |
|---|---|
| `0 0 * * * ?` | Every hour at minute 0 |
| `0 0 6 * * ?` | Every day at 06:00 |
| `0 0 0 * * ?` | Every day at midnight |
| `0 0 6 ? * MON-FRI` | Weekdays at 06:00 |

> 💡 Always check the schedule after creating a table from the UI — an hourly refresh may be more often than you need and adds serverless cost.

### View the schedule
```sql
DESCRIBE TABLE EXTENDED training.bronze.customer_lakeflow_test;
-- look at: Refresh Schedule, Last Refreshed, Latest Refresh Status, Latest Refresh (link to the pipeline update)
```

### Change the schedule
Use SQL (or the schedule option on the table page in Catalog Explorer):
```sql
-- Daily at 06:00 Amsterdam time
ALTER STREAMING TABLE training.bronze.customer_lakeflow_test
  ALTER SCHEDULE CRON '0 0 6 * * ?' AT TIME ZONE 'Europe/Amsterdam';

-- Fixed interval
ALTER STREAMING TABLE training.bronze.customer_lakeflow_test
  ALTER SCHEDULE EVERY 1 DAY;          -- HOURS 1-72, DAYS 1-31, WEEKS 1-8

-- Refresh when the upstream source changes
ALTER STREAMING TABLE training.bronze.customer_lakeflow_test
  ADD TRIGGER ON UPDATE;

-- Remove the schedule (manual / job-driven refresh only)
ALTER STREAMING TABLE training.bronze.customer_lakeflow_test
  DROP SCHEDULE;

-- Manual refresh at any time, even with a schedule
REFRESH STREAMING TABLE training.bronze.customer_lakeflow_test;
```


## 7. Table Schedule vs Job Schedule

### Does a job schedule overwrite the table schedule?
**No.** The table schedule and a job schedule are independent. If you keep both, **both run**.

Example with `training.bronze.customer_lakeflow_test`:

| Table schedule | Job schedule | Result |
|---|---|---|
| Hourly (`0 0 * * * ?`) | Daily 06:00 | 24 hourly refreshes **plus** the job's refresh at 06:00 — double work and possible overlap |
| None (`DROP SCHEDULE`) | Daily 06:00 | One refresh per day, driven by the job ✅ |
| Hourly | None | Hourly refreshes, no job needed ✅ |

> **Rule:** keep the schedule in **one place only**.

### Option A — schedule on the table (simple)
Keep `SCHEDULE` / `TRIGGER ON UPDATE` on the table (Section 6). No job required. Best when the table refreshes on its own.

### Option B — orchestrate from a Lakeflow Job (workflow)
Use this when the refresh is one step of a larger flow (e.g. ingest → transform notebook → dashboard refresh).

**Step 1 — remove the table schedule**
```sql
ALTER STREAMING TABLE training.bronze.customer_lakeflow_test DROP SCHEDULE;
```

**Step 2 — create the job with a SQL task**
* **Jobs & Pipelines → Create job → Add task → type: SQL**, select a SQL warehouse and a query containing:
```sql
REFRESH STREAMING TABLE training.bronze.customer_lakeflow_test;
```
* Or write the `REFRESH` statement in the SQL editor and click **Schedule** to create the job directly.

**Step 3 — add downstream tasks and set the job schedule**
```
Job: daily_customer_refresh   (schedule: daily 06:00 Europe/Amsterdam)
 ├── Task 1: SQL      REFRESH STREAMING TABLE training.bronze.customer_lakeflow_test
 ├── Task 2: Notebook transform to silver       (depends on Task 1)
 └── Task 3: Dashboard refresh                  (depends on Task 2)
```

### Good to know
* Each job run performs **one triggered refresh** of a standalone streaming table — it does not run continuously.
* The SQL task uses the job's SQL warehouse to submit the command; the refresh itself runs on the table's serverless pipeline.
* A Lakeflow pipeline built in the pipeline editor (like this training's `dp` pipelines) has **no default schedule**; scheduling it from the pipeline UI creates a job with a **pipeline task**.